In [ ]:
pip install pandas numpy scikit-learn xgboost mrmr-selection matplotlib seaborn

In [5]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold

from sklearn.feature_selection import (
    SelectKBest,
    f_classif,
    RFE,
    SelectFromModel
)
from sklearn.metrics import (
    roc_auc_score, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score
)

from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier
from mrmr import mrmr_classif

from sklearn.preprocessing import StandardScaler

import seaborn as sns
import matplotlib.pyplot as plt

In [6]:
df = pd.read_csv("diabetes.csv")

# Display basic information
print("Dataset Shape:", df.shape)
display(df.head())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Separate features and target
X = df.drop(columns=["Diabetes_binary"])
y = df["Diabetes_binary"]

print("\nFeature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

print("\nTarget Distribution:")
print(y.value_counts())

print("Feature Names:\n")
for i, col in enumerate(X.columns, 1):
    print(f"{i}. {col}")

Dataset Shape: (253680, 22)


,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0



Missing Values:
Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

Feature Matrix Shape: (253680, 21)
Target Shape: (253680,)

Target Distribution:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64
Feature Names:

1. HighBP
2. HighChol
3. CholCheck
4. BMI
5. Smoker
6. Stroke
7. HeartDiseaseorAttack
8. PhysActivity
9. Fruits
10. Veggies
11. HvyAlcoholConsump
12. AnyHealthcare
13. NoDocbcCost
14. GenHlth
15. MentHlth
16. PhysHlth
17. DiffWalk

In [7]:
# Number of features to evaluate
k_values = [5, 10, 15, 20]

# Feature Selection Methods
selectors = {

    "ANOVA": lambda k: SelectKBest(
        score_func=f_classif,
        k=k
    ),

    "LASSO": lambda k: SelectFromModel(
        estimator=LassoCV(cv=5),
        max_features=k,
        threshold=-np.inf
    ),

    "RandomForest": lambda k: SelectFromModel(
        estimator=RandomForestClassifier(
            random_state=42,
            n_estimators=100
        ),
        max_features=k,
        threshold=-np.inf
    )

}

In [22]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=-1),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1) 
}

In [9]:
selector_names = [
    "ANOVA",
    "LASSO",
    "RandomForest",
    "MRMR"
]


metric_names = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC"
]


all_results = {}

all_feature_frequency = {}

all_clinical_scores = {}

In [10]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [11]:
##################################################
# Clinical Relevance Baseline
# Based on established Type-2 Diabetes Risk Factors
##################################################

clinical_guidelines = {

    "BMI": 1.0,

    "HighBP": 0.95,

    "HighChol": 0.90,

    "GenHlth": 0.85,

    "Age": 0.75,

    "HeartDiseaseorAttack": 0.75,

    "PhysHlth": 0.70,

    "DiffWalk": 0.70,

    "Stroke": 0.65,

    "PhysActivity": 0.65,

    "Smoker": 0.40,

    "Sex": 0.35,

    "Income": 0.35,

    "MentHlth": 0.30,

    "HvyAlcoholConsump": 0.30,

    "Education": 0.25,

    "Fruits": 0.25,

    "Veggies": 0.25,

    "AnyHealthcare": 0.20,

    "NoDocbcCost": 0.20
}

In [12]:
##################################################
# Clinical Relevance Score Function
##################################################

def calculate_clinical_relevance(
        selected_features,
        guidelines
):

    if len(selected_features) == 0:
        return 0.0


    total_score = 0.0


    for feature in selected_features:

        score = 0.0


        for clinical_feature, weight in guidelines.items():

            if clinical_feature.lower() == feature.lower():

                score = weight
                break


        total_score += score


    return total_score / len(selected_features)

In [1]:
pip install imbalanced-learn

In [2]:
from imblearn.over_sampling import SMOTE

In [19]:
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

##################################################
# Stratified 10-Fold Cross Validation
##################################################

cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

##################################################
# SMOTE for Class Balancing
##################################################

smote = SMOTE(
    random_state=42
)

In [23]:
# Initialize placeholder structures
all_results = {k: {metric: pd.DataFrame(index=selector_names, columns=models.keys()) for metric in metric_names} for k in k_values}
all_feature_frequency = {k: {sel: {} for sel in selector_names} for k in k_values}
all_clinical_scores = {k: {sel: [] for sel in selector_names} for k in k_values}

# Dict to hold raw evaluation lists before taking the mean
raw_scores = {
    k: {
        sel: {model_name: {metric: [] for metric in metric_names} for model_name in models}
        for sel in selector_names
    }
    for k in k_values
}

print("Starting Optimized Cross-Validation Loop...")

# STEP 1: Outer loop must be the CV splits so we scale and run SMOTE ONLY ONCE per fold
for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y)):
    print(f" Processing Fold {fold_idx + 1}...")
    
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Scale once per fold
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

    # SMOTE once per fold (SMOTE creates a lot of rows, running it repeatedly wastes huge time)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
    X_train_balanced = pd.DataFrame(X_train_balanced, columns=X_train.columns)
    y_train_balanced = pd.Series(y_train_balanced)

    # STEP 2: Compute feature selections for this fold
    fold_features = {}
    for selector_name in selector_names:
        
        # PRO TIP: Compute MRMR for the MAX k value once per fold, rather than re-running it for k=5, k=10, etc.
        max_k = max(k_values)
        
        if selector_name == "MRMR":
            # MRMR scales poorly with dataset size; running it once per fold saves massive time
            selected = mrmr_classif(
                X=X_train_balanced,
                y=y_train_balanced,
                K=max_k,
                n_jobs=-1 # Use all available CPU cores to speed it up!
            )
            fold_features[selector_name] = selected
        else:
            # For standard selectors, get the max support or the underlying rank/scores if possible
            selector = selectors[selector_name](max_k)
            selector.fit(X_train_balanced, y_train_balanced)
            
            # If your selectors support ranking (like feature_importances_ or coefficients), 
            # save the sorted list so you can slice it instantly for any k
            if hasattr(selector, 'get_support'):
                fold_features[selector_name] = list(X_train_balanced.columns[selector.get_support()])
            else:
                # Fallback if your custom selectors don't support arbitrary max_k slicing
                fold_features[selector_name] = selector.selected_features_ 

    # STEP 3: Now evaluate all k variations instantly using the pre-computed features
    for k in k_values:
        for selector_name in selector_names:
            
            # Slice just the top-k features for this specific run
            features = fold_features[selector_name][:k]
            
            X_train_selected = X_train_balanced[features]
            X_test_selected = X_test_scaled[features]

            # Track clinical relevance & stability frequencies
            score = calculate_clinical_relevance(features, clinical_guidelines)
            all_clinical_scores[k][selector_name].append(score)

            for feature in features:
                all_feature_frequency[k][selector_name][feature] = (
                    all_feature_frequency[k][selector_name].get(feature, 0) + 1
                )

            # Train and evaluate models
            for model_name, model in models.items():
                model.fit(X_train_selected, y_train_balanced)
                predictions = model.predict(X_test_selected)
                probabilities = model.predict_proba(X_test_selected)[:, 1]

                raw_scores[k][selector_name][model_name]["Accuracy"].append(accuracy_score(y_test, predictions))
                raw_scores[k][selector_name][model_name]["Precision"].append(precision_score(y_test, predictions, zero_division=0))
                raw_scores[k][selector_name][model_name]["Recall"].append(recall_score(y_test, predictions, zero_division=0))
                raw_scores[k][selector_name][model_name]["F1"].append(f1_score(y_test, predictions, zero_division=0))
                raw_scores[k][selector_name][model_name]["ROC-AUC"].append(roc_auc_score(y_test, probabilities))

# STEP 4: Aggregation (Calculate means after all folds finish)
for k in k_values:
    for selector_name in selector_names:
        all_clinical_scores[k][selector_name] = np.mean(all_clinical_scores[k][selector_name])
        
        for model_name in models:
            for metric in metric_names:
                mean_score = np.mean(raw_scores[k][selector_name][model_name][metric])
                all_results[k][metric].loc[selector_name, model_name] = mean_score
                
    for metric in metric_names:
        all_results[k][metric] = all_results[k][metric].astype(float)

print("\nExperiment Complete!")

Starting Optimized Cross-Validation Loop...
 Processing Fold 1...


100%|██████████| 20/20 [00:13<00:00,  1.50it/s]


 Processing Fold 2...


100%|██████████| 20/20 [00:09<00:00,  2.17it/s]


 Processing Fold 3...


...

 Processing Fold 9...


100%|██████████| 20/20 [00:08<00:00,  2.37it/s]


 Processing Fold 10...


100%|██████████| 20/20 [00:08<00:00,  2.23it/s]



Experiment Complete!


In [24]:
for k in all_clinical_scores.keys():

    print("\n")
    print("="*70)
    print(f"Clinical Relevance Score - Top {k} Features")
    print("="*70)

    print(
        pd.DataFrame(
            all_clinical_scores[k],
            index=[f"Top-{k}"]
        )
    )



Clinical Relevance Score - Top 5 Features
       ANOVA  LASSO  RandomForest  MRMR
Top-5   0.65  0.695          0.78  0.89


Clinical Relevance Score - Top 10 Features
        ANOVA  LASSO  RandomForest   MRMR
Top-10   0.58  0.571          0.61  0.659


Clinical Relevance Score - Top 15 Features
           ANOVA     LASSO  RandomForest      MRMR
Top-15  0.543333  0.531333      0.556667  0.612333


Clinical Relevance Score - Top 20 Features
         ANOVA   LASSO  RandomForest    MRMR
Top-20  0.5275  0.5185        0.5375  0.5275


In [25]:
import pandas as pd

pd.set_option('display.max_rows', None)


for k in k_values:

    print("\n")
    print("="*80)
    print(
        f"FEATURE SELECTION STABILITY FREQUENCY (Top-{k})"
    )
    print("="*80)


    for method in all_feature_frequency[k]:

        print("\nSelector:", method)


        freq = pd.Series(
            all_feature_frequency[k][method]
        ).sort_values(
            ascending=False
        )


        print(
            freq.to_string()
        )



FEATURE SELECTION STABILITY FREQUENCY (Top-5)

Selector: ANOVA
HighBP       10
HighChol     10
CholCheck    10
BMI          10
Smoker       10

Selector: LASSO
HighBP       10
HighChol     10
CholCheck    10
BMI          10
Stroke        9
Smoker        1

Selector: RandomForest
HighBP      10
HighChol    10
BMI         10
Smoker      10
Stroke      10

Selector: MRMR
GenHlth     10
Age         10
BMI         10
HighBP      10
HighChol    10


FEATURE SELECTION STABILITY FREQUENCY (Top-10)

Selector: ANOVA
HighBP                  10
HighChol                10
CholCheck               10
BMI                     10
Smoker                  10
Stroke                  10
HeartDiseaseorAttack    10
PhysActivity            10
Fruits                  10
Veggies                 10

Selector: LASSO
HighBP                  10
HighChol                10
CholCheck               10
BMI                     10
Stroke                  10
HeartDiseaseorAttack    10
PhysActivity            10
Fruits    

In [26]:
##################################################
# Print Model Performance Results
##################################################

for k in k_values:

    print("\n")
    print("="*90)
    print(f"MODEL PERFORMANCE RESULTS - TOP {k} FEATURES")
    print("="*90)


    for metric in metric_names:

        print("\n")
        print("-"*70)
        print(metric)
        print("-"*70)


        print(
            all_results[k][metric]
        )



MODEL PERFORMANCE RESULTS - TOP 5 FEATURES


----------------------------------------------------------------------
Accuracy
----------------------------------------------------------------------
              Logistic Regression  Random Forest   XGBoost
ANOVA                    0.692664       0.855889  0.851321
LASSO                    0.697126       0.856528  0.849180
RandomForest             0.696827       0.857458  0.851821
MRMR                     0.727275       0.860446  0.863974


----------------------------------------------------------------------
Precision
----------------------------------------------------------------------
              Logistic Regression  Random Forest   XGBoost
ANOVA                    0.272692       0.460021  0.436839
LASSO                    0.275392       0.465682  0.430499
RandomForest             0.274879       0.469040  0.438434
MRMR                     0.306596       0.497767  0.534314


--------------------------------------------------------